# Credit Risk Scoring & Expected Loss Analysis
## Notebook 3: SQL Risk Analysis

This notebook loads the cleaned dataset into an in-memory SQLite database to simulate querying a relational banking database. SQL aggregations analyse demographic and behavioural risk profiles.

## 1. Setup

In [1]:
import pandas as pd
import sqlite3

PROCESSED_PATH = '../data/processed/cleaned_credit_card_data.csv'
df = pd.read_csv(PROCESSED_PATH)

conn = sqlite3.connect(':memory:')
df.to_sql('credit_portfolio', conn, index=False, if_exists='replace')
print(f'Loaded {len(df):,} rows into SQLite table: credit_portfolio')

Loaded 30,000 rows into SQLite table: credit_portfolio


## 2. Query 1: Default Rate by Education Level

Examines whether educational attainment correlates with default risk and average approved credit limit.

Education codes: 1 = Graduate School, 2 = University, 3 = High School, 4 = Other

In [2]:
query_education = """
SELECT
    CASE EDUCATION
        WHEN 1 THEN 'Graduate School'
        WHEN 2 THEN 'University'
        WHEN 3 THEN 'High School'
        WHEN 4 THEN 'Other'
    END                                        AS Education_Level,
    COUNT(*)                                   AS Customer_Count,
    ROUND(AVG(LIMIT_BAL), 0)                   AS Avg_Credit_Limit,
    ROUND(AVG(DEFAULT_NEXT_MONTH) * 100, 2)    AS Default_Rate_Pct
FROM credit_portfolio
GROUP BY EDUCATION
ORDER BY Default_Rate_Pct DESC;
"""
result_education = pd.read_sql(query_education, conn)
print('Default Rate by Education Level:')
print(result_education.to_string(index=False))
result_education

Default Rate by Education Level:
Education_Level  Customer_Count  Avg_Credit_Limit  Default_Rate_Pct
    High School            4917          126550.0             25.16
     University           14030          147062.0             23.73
Graduate School           10585          212956.0             19.23
          Other             468          181316.0              7.05


,Education_Level,Customer_Count,Avg_Credit_Limit,Default_Rate_Pct
0,High School,4917,126550.0,25.16
1,University,14030,147062.0,23.73
2,Graduate School,10585,212956.0,19.23
3,Other,468,181316.0,7.05


**Finding**: High school educated clients carry the highest default rate, while graduate-school clients have both higher credit limits and lower default rates. The "Other" category is small (468 clients) and shows the lowest default rate.

## 3. Query 2: Default Rate by Repayment Status (PAY_0)

The most recent repayment status (`PAY_0`) is a critical early-warning signal in credit portfolios.

In [3]:
query_pay0 = """
SELECT
    PAY_0                                      AS Repayment_Status,
    COUNT(*)                                   AS Customer_Count,
    ROUND(AVG(DEFAULT_NEXT_MONTH) * 100, 2)    AS Default_Rate_Pct,
    ROUND(AVG(LIMIT_BAL), 0)                   AS Avg_Credit_Limit,
    ROUND(AVG(BILL_AMT1), 0)                   AS Avg_Bill_Amount
FROM credit_portfolio
GROUP BY PAY_0
ORDER BY PAY_0 ASC;
"""
result_pay0 = pd.read_sql(query_pay0, conn)
print('Default Rate by Repayment Status (PAY_0):')
print(result_pay0.to_string(index=False))
result_pay0

Default Rate by Repayment Status (PAY_0):
 Repayment_Status  Customer_Count  Default_Rate_Pct  Avg_Credit_Limit  Avg_Bill_Amount
               -2            2759             13.23          250391.0           8793.0
               -1            5686             16.78          206769.0          10485.0
                0           14737             12.81          154265.0          78418.0
                1            3688             33.95          148796.0          27153.0
                2            2667             69.14          111495.0          64888.0
                3             322             75.78           81553.0          37820.0
                4              76             68.42           78158.0          68101.0
                5              26             50.00           73077.0          81035.0
                6              11             54.55          100909.0          87588.0
                7               9             77.78          145556.0         155073.0
 

,Repayment_Status,Customer_Count,Default_Rate_Pct,Avg_Credit_Limit,Avg_Bill_Amount
0,-2,2759,13.23,250391.0,8793.0
1,-1,5686,16.78,206769.0,10485.0
2,0,14737,12.81,154265.0,78418.0
3,1,3688,33.95,148796.0,27153.0
4,2,2667,69.14,111495.0,64888.0
5,3,322,75.78,81553.0,37820.0
6,4,76,68.42,78158.0,68101.0
7,5,26,50.00,73077.0,81035.0
8,6,11,54.55,100909.0,87588.0
9,7,9,77.78,145556.0,155073.0


**Finding**: Default rates rise sharply at PAY_0 ≥ 2 (2+ month delay). This confirms PAY_0 as the dominant risk signal in the dataset.

## 4. Query 3: High-Risk Segment Identification (CTE + Window Function)

A Common Table Expression (CTE) with `NTILE(4)` ranks clients into bill-amount quartiles — a technique commonly used in credit analytics portfolio segmentation.

In [4]:
query_cte = """
WITH client_risk AS (
    SELECT
        LIMIT_BAL,
        BILL_AMT1,
        DEFAULT_NEXT_MONTH,
        NTILE(4) OVER (ORDER BY BILL_AMT1 DESC) AS Bill_Quartile
    FROM credit_portfolio
)
SELECT
    Bill_Quartile,
    COUNT(*)                                   AS Customer_Count,
    ROUND(AVG(LIMIT_BAL), 0)                   AS Avg_Credit_Limit,
    ROUND(AVG(BILL_AMT1), 0)                   AS Avg_Bill_Amount,
    ROUND(AVG(DEFAULT_NEXT_MONTH) * 100, 2)    AS Default_Rate_Pct
FROM client_risk
GROUP BY Bill_Quartile
ORDER BY Bill_Quartile;
"""
result_cte = pd.read_sql(query_cte, conn)
result_cte['Quartile_Label'] = result_cte['Bill_Quartile'].map({
    1: 'Q1 Highest Bill', 2: 'Q2', 3: 'Q3', 4: 'Q4 Lowest Bill'
})
print('Risk Profile by Bill Amount Quartile (CTE + NTILE):')
print(result_cte[['Quartile_Label','Customer_Count','Avg_Credit_Limit',
                   'Avg_Bill_Amount','Default_Rate_Pct']].to_string(index=False))
result_cte

Risk Profile by Bill Amount Quartile (CTE + NTILE):
 Quartile_Label  Customer_Count  Avg_Credit_Limit  Avg_Bill_Amount  Default_Rate_Pct
Q1 Highest Bill            7500          213990.0         150564.0             20.04
             Q2            7500          118417.0          41755.0             22.51
             Q3            7500          140950.0          11764.0             22.93
 Q4 Lowest Bill            7500          196580.0            810.0             23.00


,Bill_Quartile,Customer_Count,Avg_Credit_Limit,Avg_Bill_Amount,Default_Rate_Pct,Quartile_Label
0,1,7500,213990.0,150564.0,20.04,Q1 Highest Bill
1,2,7500,118417.0,41755.0,22.51,Q2
2,3,7500,140950.0,11764.0,22.93,Q3
3,4,7500,196580.0,810.0,23.00,Q4 Lowest Bill


**Finding**: The relationship between bill amount and default rate is non-linear. Repayment behaviour (PAY_0) remains a stronger driver than outstanding bill amount.

## 5. Query 4: Portfolio Exposure Concentration

Portfolio concentration analysis identifies the credit-limit bands where potential losses are concentrated — a standard exercise in credit risk management.

In [5]:
query_concentration = """
SELECT
    CASE
        WHEN LIMIT_BAL < 100000  THEN 'A: Under 100K'
        WHEN LIMIT_BAL < 300000  THEN 'B: 100K to 300K'
        WHEN LIMIT_BAL < 500000  THEN 'C: 300K to 500K'
        ELSE                          'D: 500K+'
    END                                        AS Credit_Limit_Band,
    COUNT(*)                                   AS Customer_Count,
    ROUND(SUM(LIMIT_BAL) / 1000000.0, 1)      AS Total_Exposure_M_NTD,
    ROUND(AVG(DEFAULT_NEXT_MONTH) * 100, 2)    AS Default_Rate_Pct
FROM credit_portfolio
GROUP BY Credit_Limit_Band
ORDER BY Credit_Limit_Band;
"""
result_concentration = pd.read_sql(query_concentration, conn)
print('Portfolio Exposure Concentration by Credit Limit Band:')
print(result_concentration.to_string(index=False))
result_concentration

Portfolio Exposure Concentration by Credit Limit Band:
Credit_Limit_Band  Customer_Count  Total_Exposure_M_NTD  Default_Rate_Pct
    A: Under 100K           11450                 554.9             29.80
  B: 100K to 300K           13433                2453.2             18.79
  C: 300K to 500K            4189                1532.3             14.23
         D: 500K+             928                 484.3             11.21


,Credit_Limit_Band,Customer_Count,Total_Exposure_M_NTD,Default_Rate_Pct
0,A: Under 100K,11450,554.9,29.80
1,B: 100K to 300K,13433,2453.2,18.79
2,C: 300K to 500K,4189,1532.3,14.23
3,D: 500K+,928,484.3,11.21


In [6]:
conn.close()
print('SQLite connection closed.')

SQLite connection closed.


## 6. SQL Analysis Summary

| Query | Technique | Key Finding |
|---|---|---|
| Default by Education | GROUP BY + CASE WHEN | Graduate-school clients have lowest default rates |
| Default by PAY_0 | GROUP BY | Sharp default spike at ≥2-month payment delay |
| Bill Quartiles | CTE + NTILE window function | High-bill quartile has higher credit limits; non-linear default relationship |
| Exposure Bands | CASE WHEN + SUM | Majority of exposure concentrated in 100K–300K band |

These queries demonstrate: aggregation, conditional labelling (`CASE WHEN`), Common Table Expressions (CTEs), and window functions (`NTILE`) — standard SQL tools in banking analytics.